In [2]:
import os
import urllib.request

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

DATA_PATH = "titanic.csv"
DATA_URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"


def load_data(path: str = DATA_PATH) -> pd.DataFrame:
    """Load the Titanic dataset from a local file, downloading it first if needed."""
    if not os.path.exists(path):
     urllib.request.urlretrieve(DATA_URL, path)
    df = pd.read_csv(path)
    return df

df = load_data()

print("=" * 60)
print("STEP 1: DATA LOADED")
print("=" * 60)
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head())
print()
print("=" * 60)
print("STEP 2: PREPROCESSING")
print("=" * 60)

df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"], errors="ignore")

df["Age"] = df["Age"].fillna(df["Age"].median())
if "Embarked" in df.columns:
    df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
if "Fare" in df.columns:
    df["Fare"] = df["Fare"].fillna(df["Fare"].median())

df["Sex"] = LabelEncoder().fit_transform(df["Sex"])
if "Embarked" in df.columns:
    df["Embarked"] = LabelEncoder().fit_transform(df["Embarked"])

print("Missing values remaining per column:")
print(df.isnull().sum())
print()
print("Cleaned data preview:")
print(df.head())
print()


print("=" * 60)
print("STEP 3: TRAIN/TEST SPLIT")
print("=" * 60)

X = df.drop(columns=["Survived"])
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} passengers")
print(f"Testing set:  {X_test.shape[0]} passengers")
print()

print("=" * 60)
print("STEP 4: TRAINING THE MODEL")
print("=" * 60)

model = DecisionTreeClassifier(max_depth=5, random_state=42)
model.fit(X_train, y_train)
print("Decision Tree Classifier trained.")
print()

print("=" * 60)
print("STEP 5: EVALUATION")
print("=" * 60)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")
print()
print("Classification report:")
print(classification_report(y_test, y_pred, target_names=["Did not survive", "Survived"]))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))
print()

importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Feature importance:")
print(importances)

STEP 1: DATA LOADED
Shape: 891 rows, 12 columns
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      